In [1]:
import csv
import json
import math
import pickle
from pathlib import Path
from functools import partial

import torch
from torch.utils.data import DataLoader

import lm.ngram as ngram
import lm.evaluation as evaluation
import lm.lstm as lstm

from config import PROCESSED_DIR, ARTIFACT_DIR, RESULT_DIR

DATA_DIR = PROCESSED_DIR

ARTIFACT_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)


def load_json(filename):
    with (DATA_DIR / filename).open(encoding="utf-8") as file:
        return json.load(file)


train_documents = load_json("train.json")
test_documents = load_json("test.json")
vocab_list = load_json("vocab.json")
vocab = set(vocab_list)

checkpoint = torch.load(
    ARTIFACT_DIR / "lstm_best.pt",
    map_location="cpu",
    weights_only=True,
)

assert vocab_list == checkpoint["vocab_list"]

results = []

In [2]:
for name, n, k in [
    ("Bigram", 2, 0.01),
    ("Trigram", 3, 0.001),
]:
    counts, contexts = ngram.count_ngrams(
        train_documents, n=n
    )

    score = evaluation.perplexity(
        documents=test_documents,
        n=n,
        ngram_counts=counts,
        context_counts=contexts,
        vocab=vocab,
        k=k,
    )

    results.append({
        "model": name,
        "test_perplexity": score,
    })

    with (ARTIFACT_DIR / f"{name.lower()}.pkl").open("wb") as file:
        pickle.dump({
            "n": n,
            "k": k,
            "ngram_counts": counts,
            "context_counts": contexts,
            "vocab_list": vocab_list,
        }, file, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"{name} test perplexity: {score:.2f}")

Bigram test perplexity: 387.81
Trigram test perplexity: 2220.55


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

token_to_id = {
    token: index
    for index, token in enumerate(vocab_list)
}

model = lstm.LSTMLanguageModel(
    **checkpoint["model_config"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_ids = [
    [token_to_id[token] for token in document["tokens"]]
    for document in test_documents
]

test_inputs, test_targets = lstm.create_sequences(
    test_ids,
    bos_id=token_to_id["<bos>"],
    eos_id=token_to_id["<eos>"],
    sequence_length=checkpoint["sequence_length"],
)

expected_count = sum(
    len(document["tokens"]) + 1
    for document in test_documents
)

assert sum(len(y) for y in test_targets) == expected_count

test_loader = DataLoader(
    list(zip(test_inputs, test_targets)),
    batch_size=32,
    shuffle=False,
    collate_fn=partial(
        lstm.collate_batch,
        input_pad_id=token_to_id["<eos>"],
    ),
    num_workers=0,
)

test_loss = lstm.run_epoch(model, test_loader, device)
lstm_ppl = math.exp(test_loss)

results.append({
    "model": "LSTM",
    "test_perplexity": lstm_ppl,
})

print("Cihaz:", device)
print("Seçilen LSTM epoch'u:", checkpoint["epoch"])
print("Değerlendirilen hedef sayısı:", expected_count)
print(f"LSTM test perplexity: {lstm_ppl:.2f}")

Cihaz: cuda
Seçilen LSTM epoch'u: 7
Değerlendirilen hedef sayısı: 100003
LSTM test perplexity: 171.62


In [4]:
print(f"{'Model':<12} {'Test perplexity':>18}")
print("-" * 31)

for row in results:
    print(
        f"{row['model']:<12} "
        f"{row['test_perplexity']:>18.2f}"
    )

with (RESULT_DIR / "test_results.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["model", "test_perplexity"],
    )
    writer.writeheader()
    writer.writerows(results)

Model           Test perplexity
-------------------------------
Bigram                   387.81
Trigram                 2220.55
LSTM                     171.62
